In [59]:
import os
import pandas as pd
from IPython.display import display
from tqdm import tqdm
import re

folder = os.getcwd()
excels = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".xlsx")]

In [60]:
dfs = []
for i, pth in enumerate(tqdm(excels)):
    df_i = pd.read_excel(pth)
    df_i = df_i.loc[:, ['Part Number', 'Description', 'Models']]
    dfs.append(df_i)

df_all = pd.concat(dfs, ignore_index=True)

100%|██████████| 19/19 [00:02<00:00,  8.55it/s]


In [61]:
df_all[df_all['Part Number'] == '603']

,Part Number,Description,Models


In [62]:
def join_ordered(series: pd.Series):
    all_parts = []

    for item in series.dropna(): 
        parts = str(item).split(';') 
        for p in parts:
            p_clean = re.sub(r'\s+', '', p).upper()
            if p_clean:
                all_parts.append(p_clean)

    seen = set()
    result = []
    for v in all_parts:
        if v not in seen:
            seen.add(v)
            result.append(v)

    return "; ".join(result) if result else None


def join_ordered_description(series: pd.Series):
    all_parts = []

    for item in series.dropna():
        text = str(item).strip()
        if not text:
            continue

        parts = [p.strip() for p in text.split(";")]

        for p in parts:
            p_clean = re.sub(r"\s+", " ", p).strip()
            if p_clean:
                all_parts.append(p_clean)

    seen = set()
    result = []

    for v in all_parts:
        key = re.sub(r"\s+", " ", v).strip().lower()

        if key not in seen:
            seen.add(key)
            result.append(v)

    return "; ".join(result) if result else None


In [63]:
def merge_parts(df):
    df = df.copy()

    agg_map = {
        "Description": join_ordered_description,
        "Models": join_ordered
    }

    df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

    return df_agg

In [64]:
df_all.info()

df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

df_letters = df_all[part_numeric.isna()].copy()
df_numbers = df_all[part_numeric.notna()].copy()

if not df_numbers.empty:
    df_numbers["Part Number"] = (
        part_numeric[part_numeric.notna()]
        .astype(int)
        .astype(str)
        .str.zfill(8)
    )

<class 'pandas.DataFrame'>
RangeIndex: 15698 entries, 0 to 15697
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Part Number  15698 non-null  object
 1   Description  15698 non-null  str   
 2   Models       15698 non-null  str   
dtypes: object(1), str(2)
memory usage: 368.1+ KB


In [65]:
df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)

In [66]:
df_merged.to_excel(r"final_модели.xlsx", sheet_name="Parts", index=False)